# Semantic Chunking

The goal is to split documents into semantically coherent chunks by embedding sentences and cutting where cosine similarity between adjacent sentences drops below a threshold.

The default model is `BAAI/bge-base-en-v1.5`. This model is midrange (768 dimensions) and offers a good balance between semantic quality and computational efficiency (can be run locally). The base variant is large enough to capture nuanced topic shifts, but still lightweight to run on my computer. Also, it is open-source and does not require paid API access.

You can change the model name to any model from Hugging Face to see how the chunking differs. Specifically, you can use any model on the MTEB Leaderboard: https://huggingface.co/spaces/mteb/leaderboard. The best model that I can still run on my computer seems to be `BAAI/bge-large-en-v1.5` but `all-mpnet-base-v2` seems to be good too.

You can also change the similarity threshold to decide when to split a piece of text. Here's what each threshold means:

Higher (0.7–0.8) :	More aggressive splitting, smaller chunks, only very similar sentences stay grouped<br>
0.5 (default) :	Moderate splitting<br>
Lower (0.3–0.4)	: Less splitting, bigger chunks, only major topic shifts cause a break<br>

In [18]:
import os
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

## Parameters

In [ ]:
MODEL_NAME = "BAAI/bge-large-en-v1.5"
SIMILARITY_THRESHOLD = 0.5

DOC_FILES = [
    "document1_idx2_how_should_you_be.txt",
    "document2_idx3_kerfuffle.txt",
    "document3_idx4_laying.txt",
    "document4_idx5_strife.txt",
    "document5_idx6_russia.txt",
]

## Load Documents

In [20]:
documents = {}
for fname in DOC_FILES:
    with open(fname, "r", encoding="utf-8") as f:
        documents[fname] = f.read()

print(f"Loaded {len(documents)} documents")
for name, text in documents.items():
    print(f"  {name}: {len(text):,} chars")

Loaded 5 documents
  document1_idx2_how_should_you_be.txt: 12,900 chars
  document2_idx3_kerfuffle.txt: 5,714 chars
  document3_idx4_laying.txt: 44,682 chars
  document4_idx5_strife.txt: 13,145 chars
  document5_idx6_russia.txt: 8,112 chars


## Chunking Functions

In [ ]:
def split_sentences(text: str) -> list[str]:
    """Tokenize text into sentences using NLTK."""
    sentences = sent_tokenize(text)
    return [s.strip() for s in sentences if s.strip()]


def embed_sentences(sentences: list[str], model: SentenceTransformer) -> np.ndarray:
    """Return (n_sentences, dim) embedding matrix."""
    return model.encode(sentences, show_progress_bar=False)


def semantic_chunk(
    sentences: list[str], embeddings: np.ndarray, threshold: float
) -> list[list[str]]:
    """Group sentences into chunks, splitting where adjacent similarity < threshold."""
    if len(sentences) == 0:
        return []
    if len(sentences) == 1:
        return [sentences]

    chunks = []
    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine_similarity(
            embeddings[i - 1].reshape(1, -1), embeddings[i].reshape(1, -1)
        )[0, 0]
        if sim >= threshold:
            current_chunk.append(sentences[i])
        else:
            chunks.append(current_chunk)
            current_chunk = [sentences[i]]

    chunks.append(current_chunk)
    return chunks

## Run Chunking on All Documents

In [ ]:
model = SentenceTransformer(MODEL_NAME)

results = []
for fname, text in documents.items():
    sentences = split_sentences(text)
    embeddings = embed_sentences(sentences, model)
    chunks = semantic_chunk(sentences, embeddings, SIMILARITY_THRESHOLD)
    results.append({
        "filename": fname,
        "sentences": sentences,
        "chunks": chunks,
    })
    print(f"{fname}: {len(sentences)} sentences -> {len(chunks)} chunks")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5153.06it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


document1_idx2_how_should_you_be.txt: 86 sentences -> 17 chunks
document2_idx3_kerfuffle.txt: 57 sentences -> 28 chunks
document3_idx4_laying.txt: 576 sentences -> 288 chunks
document4_idx5_strife.txt: 64 sentences -> 15 chunks
document5_idx6_russia.txt: 63 sentences -> 29 chunks


## Summary Statistics

In [23]:
all_chunks = [c for r in results for c in r["chunks"]]
chunk_word_counts = [sum(len(s.split()) for s in c) for c in all_chunks]
chunk_sent_counts = [len(c) for c in all_chunks]
chunks_per_doc = [len(r["chunks"]) for r in results]

print("=" * 50)
print("SEMANTIC CHUNKING SUMMARY")
print("=" * 50)
print(f"Model:                   {MODEL_NAME}")
print(f"Similarity threshold:    {SIMILARITY_THRESHOLD}")
print(f"Documents processed:     {len(results)}")
print()
print(f"Avg chunks/document:     {np.mean(chunks_per_doc):.1f}")
print(f"Avg words/chunk:         {np.mean(chunk_word_counts):.1f}")
print(f"Avg sentences/chunk:     {np.mean(chunk_sent_counts):.1f}")
print()
print(f"Min chunk length (words):      {min(chunk_word_counts)}")
print(f"Max chunk length (words):      {max(chunk_word_counts)}")
print(f"Min chunk length (sentences):  {min(chunk_sent_counts)}")
print(f"Max chunk length (sentences):  {max(chunk_sent_counts)}")
print()
print("Chunks per document:")
for r in results:
    print(f"  {r['filename']}: {len(r['chunks'])}")

SEMANTIC CHUNKING SUMMARY
Model:                   BAAI/bge-base-en-v1.5
Similarity threshold:    0.5
Documents processed:     5

Avg chunks/document:     75.4
Avg words/chunk:         39.6
Avg sentences/chunk:     2.2

Min chunk length (words):      1
Max chunk length (words):      442
Min chunk length (sentences):  1
Max chunk length (sentences):  17

Chunks per document:
  document1_idx2_how_should_you_be.txt: 17
  document2_idx3_kerfuffle.txt: 28
  document3_idx4_laying.txt: 288
  document4_idx5_strife.txt: 15
  document5_idx6_russia.txt: 29


## Example Chunks

In [24]:
EXAMPLE_CHUNKS_PER_DOC = 4

for r in results:
    print("=" * 60)
    print(f"Document: {r['filename']}")
    print("=" * 60)
    for i, chunk in enumerate(r["chunks"][:EXAMPLE_CHUNKS_PER_DOC]):
        chunk_text = " ".join(chunk)
        print(f"\n--- Chunk {i + 1} ({len(chunk)} sentence(s), {len(chunk_text.split())} words) ---")
        print(chunk_text)
    print()

Document: document1_idx2_how_should_you_be.txt

--- Chunk 1 (5 sentence(s), 66 words) ---
How Should You Be? Try Taking Suggestions. More than offering prescriptions, suggestions \'97 good or bad \'97 about how we should live broaden our worldview and put us in touch with our desires. Some chapters of life attract advice. After a breakup, friends who are typically silent about your romances might urge you to get back on dating apps and block your ex on social media.

--- Chunk 2 (17 sentence(s), 351 words) ---
Starting a family gives rise to opinions about prenatal vitamins and stroller brands; interviewing for a new job might prompt advice about salary negotiations and work-life balance. Recently, I entered a suggestion-heavy era. A long relationship had ended, and I pondered leaving my job and moving to a different neighborhood. Suddenly, I was in need of input from others on how to find a new laundromat, enroll in health insurance as a freelancer, live alone and navigate online dati

## Save Chunks to Files

Write all chunks for each document to a `.txt` file in `chunking_output/`, separated by a clear delimiter.

In [25]:
OUTPUT_DIR = "chunking_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for r in results:
    base = os.path.splitext(r["filename"])[0]
    out_path = os.path.join(OUTPUT_DIR, f"{base}_chunks.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        for i, chunk in enumerate(r["chunks"]):
            chunk_text = " ".join(chunk)
            f.write(f"=== Chunk {i + 1} ({len(chunk)} sentence(s), {len(chunk_text.split())} words) ===\n")
            f.write(chunk_text + "\n\n")
    print(f"Saved {len(r['chunks'])} chunks -> {out_path}")

Saved 17 chunks -> chunking_output/document1_idx2_how_should_you_be_chunks.txt
Saved 28 chunks -> chunking_output/document2_idx3_kerfuffle_chunks.txt
Saved 288 chunks -> chunking_output/document3_idx4_laying_chunks.txt
Saved 15 chunks -> chunking_output/document4_idx5_strife_chunks.txt
Saved 29 chunks -> chunking_output/document5_idx6_russia_chunks.txt
